# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object and print summary
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Dataset identifier: {md.identifier}")
print(f"License: {md.license}")
print(f"Date published: {md.datePublished}")
print(f"Authors: {[a['@id'] for a in md.author]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list the all record sets, their fields, and available columns. All references are made by their `@id`.

In [ ]:
# List record sets and their fields/columns by @id
record_sets = getattr(md, 'recordSet', [])
if not record_sets:
    print('No record sets found in metadata.')
else:
    for rs in record_sets:
        print(f"- Record set @id: {rs['@id']}")
        # Print columns if available
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        print(f"  Columns (@id): {[c['@id'] for c in columns]}")
        # Print fields if available
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields (@id): {[f['@id'] for f in fields]}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> **Note:** For the FAIR² dataset, the primary record set containing protein information typically has an `@id` like `'cr:RecordSet/proteins'` (this will be auto-discovered by the code in Section 2 on real schemas). Adjust as needed for your schema.

In [ ]:
# Find available record set @ids
record_sets_list = []
# Collect discovered record set @ids only
for rs in getattr(md, 'recordSet', []):
    record_sets_list.append(rs['@id'])
print(f"All available record sets @id: {record_sets_list}")

# Attempt to load all record sets
dataframes = {}
for rsid in record_sets_list:
    print(f"Loading records from record set {rsid}")
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
    else:
        print(f"No records found for {rsid}.")

# Pick the first available record set with data
if dataframes:
    first_rsid = next(iter(dataframes))
    print(f"Fields/columns in {first_rsid}:")
    print(dataframes[first_rsid].columns.tolist())
    display(dataframes[first_rsid].head())
else:
    print('No data available in any record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For example, we filter by molecular weight (MW) if available, normalize it, and group by accession (or another field).

In [ ]:
import numpy as np
# Select your field IDs (adjust according to actual @ids and DataFrame columns--inspect Section 3 output)

# Example mapping: suppose MW (Molecular Weight) field is named 'cr:Field/mw'
record_set_id = first_rsid if dataframes else None
# Try to find a numeric field in columns
numeric_candidates = []
if record_set_id:
    cols = dataframes[record_set_id].columns
    for c in cols:
        if any(s in c.lower() for s in ['weight', 'mw', 'abundance', 'count', 'peptide', 'coverage', 'score', 'amount']):
            # Heuristic: likely numeric
            numeric_candidates.append(c)

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field '{numeric_field_id}' for analysis.")
    # Ensure numeric type
    try:
        dataframes[record_set_id][numeric_field_id] = pd.to_numeric(dataframes[record_set_id][numeric_field_id], errors='coerce')
    except Exception as e:
        print(e)
    threshold = dataframes[record_set_id][numeric_field_id].quantile(0.75)  # For demo, filter top quartile
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (top quartile):")
    display(filtered_df.head())

    # Normalize
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, field_norm]].head())

    # Group by another field, e.g., accession
    group_candidates = [c for c in cols if any(k in c.lower() for k in ['accession', 'group', 'category', 'id'])]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('No suitable numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot a histogram of the molecular weight or abundance field.

In [ ]:
import matplotlib.pyplot as plt

if record_set_id and numeric_candidates:
    plt.figure(figsize=(8, 5))
    series = dataframes[record_set_id][numeric_field_id].dropna()
    plt.hist(series, bins=30, color='dodgerblue', edgecolor='black', alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset with `mlcroissant`, listing all record sets and fields by `@id`, performing data extraction, and running basic processing and visualization on the molecular data. For robust, reproducible scientific analysis, always use the `@id` references for record sets and fields as provided in the Croissant schema.